07/11/2026
Este notebook tendra la responsabilidad de preparar el ambiente con arquitectura medallion y acceso con external location - se crean condiciones para proteger los datos y que no se borren despues de cerrar los procesos


In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "pysmartdata2026";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%python
# PROTECCION: las celdas de borrado (DROP CATALOG, DROP SCHEMA, dbutils.fs.rm) solo se ejecutan si este parametro es 'SI'.
# Por defecto es 'NO', asi que ejecutar todo el notebook NO borra datos.
dbutils.widgets.dropdown("reiniciar_ambiente", "NO", ["NO", "SI"], "Reiniciar ambiente (BORRA TODOS LOS DATOS)")
reiniciar_ambiente = dbutils.widgets.get("reiniciar_ambiente") == "SI"
print("ATENCION: el ambiente se REINICIARA y se borraran los datos" if reiniciar_ambiente else "Modo seguro: no se borrara ningun dato")

Modo seguro: no se borrara ningun dato


In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%python
# PROTEGIDO: solo se ejecuta con reiniciar_ambiente = 'SI'
if reiniciar_ambiente:
    spark.sql("DROP CATALOG IF EXISTS catalog_pysmartdata CASCADE")
    print("Catalogo catalog_pysmartdata eliminado")
else:
    print("Omitido: DROP CATALOG (reiniciar_ambiente = NO)")

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_pysmartdata
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%python
# PROTEGIDO: solo se ejecuta con reiniciar_ambiente = 'SI'
if reiniciar_ambiente:
    for esquema in ["raw", "bronze", "silver", "golden"]:
        spark.sql(f"DROP SCHEMA IF EXISTS catalog_pysmartdata.{esquema}")
    print("Esquemas eliminados")
else:
    print("Omitido: DROP SCHEMA (reiniciar_ambiente = NO)")

In [0]:
%python
# PROTEGIDO: solo se ejecuta con reiniciar_ambiente = 'SI'
if reiniciar_ambiente:
    dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
    dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
    dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)
    print("Contenedores bronze, silver y golden vaciados")
else:
    print("Omitido: dbutils.fs.rm de bronze/silver/golden (reiniciar_ambiente = NO)")

True

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.raw;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.silver;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.bronze.facturacion (
periodo integer,
suscriptor integer,
nombre_cliente string,
direccion string, 
estrato INTEGER,
codigo_concepto INTEGER,
concepto string,
consumo_m3 integer, 
valor_cargo_fijo double, 
valor_consumo double,  
valor_total double,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/facturacion"

In [0]:
%sql
-- ALTER TABLE catalog_pysmartdata.bronze.facturacion
--   ADD COLUMNS (valor_total DOUBLE COMMENT 'Valor total facturado (columna TOTAL del archivo)' AFTER valor_consumo);

In [0]:
DESCRIBE TABLE catalog_pysmartdata.bronze.facturacion

col_name,data_type,comment
periodo,int,null
suscriptor,int,null
nombre_cliente,string,null
direccion,string,null
estrato,int,null
codigo_concepto,int,null
concepto,string,null
consumo_m3,int,null
valor_cargo_fijo,double,null
valor_consumo,double,null


In [0]:

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.bronze.recaudos (
suscriptor integer,
periodo integer,
entidad string,
descripcion string, 
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date, 
codigo_tarifa integer,
codigo_concepto integer, 
concepto string ,  
valor_pagado double,    
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/recaudos"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.facturacion_transformed (
periodo integer,
suscriptor integer,
nombre_cliente string,
direccion string, 
estrato INTEGER,
codigo_concepto INTEGER,
concepto string,
consumo_m3 integer, 
valor_cargo_fijo double, 
valor_consumo double,   
valor_total double,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/facturacion_transformed"

In [0]:
%sql
-- ALTER TABLE catalog_pysmartdata.silver.facturacion_transformed
--   ADD COLUMNS (valor_total DOUBLE COMMENT 'Valor total facturado (columna TOTAL del archivo)' AFTER valor_consumo);

In [0]:
DESCRIBE TABLE catalog_pysmartdata.silver.facturacion_transformed

col_name,data_type,comment
periodo,int,null
suscriptor,int,null
nombre_cliente,string,null
direccion,string,null
estrato,int,null
codigo_concepto,int,null
concepto,string,null
consumo_m3,int,null
valor_cargo_fijo,double,null
valor_consumo,double,null


In [0]:
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.facturacion_rechazados (
periodo integer,
suscriptor integer,
nombre_cliente string,
direccion string, 
estrato INTEGER,
codigo_concepto INTEGER,
concepto string,
consumo_m3 integer, 
valor_cargo_fijo double, 
valor_consumo double,   
valor_total double,
ingestion_date timestamp,
motivos_rechazo array<string> COMMENT 'Reglas de calidad que incumple el registro',
fecha_validacion timestamp COMMENT 'Fecha y hora en que se ejecutó la validación'
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/facturacion_rechazados"
COMMENT 'registros de facturación que no pasaron las reglas de calidad y deben ser validados por los analista de negocio'

In [0]:
DESCRIBE TABLE catalog_pysmartdata.silver.facturacion_rechazados

col_name,data_type,comment
periodo,int,null
suscriptor,int,null
nombre_cliente,string,null
direccion,string,null
estrato,int,null
codigo_concepto,int,null
concepto,string,null
consumo_m3,int,null
valor_cargo_fijo,double,null
valor_consumo,double,null


In [0]:

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.recaudos_transformed (
suscriptor integer,
periodo integer,
entidad string,
descripcion string, 
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date, 
codigo_tarifa integer,
codigo_concepto integer, 
concepto string ,  
valor_pagado double,    
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/recaudos_transformed"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.recaudos_rechazados (
suscriptor integer,
periodo integer,
entidad string,
descripcion string, 
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date, 
codigo_tarifa integer,
codigo_concepto integer, 
concepto string,  
valor_pagado double,    
ingestion_date timestamp,
motivos_rechazo array<string> COMMENT 'Reglas de calidad que incumple el registro',
fecha_validacion timestamp COMMENT 'Fecha y hora en que se ejecutó la validación'
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/recaudos_rechazados"
COMMENT 'registros de recaudos que no pasaron las reglas de calidad y deben ser validados por los analistas de negocio'

In [0]:
%sql
DESCRIBE TABLE catalog_pysmartdata.silver.recaudos_rechazados

col_name,data_type,comment
suscriptor,int,null
periodo,int,null
entidad,string,null
descripcion,string,null
codigo_punto_recaudo,int,null
descripcion_punto_recaudo,string,null
fecha_pago,date,null
codigo_tarifa,int,null
codigo_concepto,int,null
concepto,string,null


###Tablas Golden

In [0]:
-- /*KPI DE FACTURACION POR PERIOD*/

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.golden.kpi_facturacion_periodo (
periodo integer COMMENT 'Periodo de facturación (yyyyMM)',
estrato integer COMMENT 'Estrato socioeconómico (null = sin estrato)',
codigo_concepto integer,
concepto string,
num_suscriptores bigint COMMENT 'Suscriptores distintos facturados',
consumo_total_m3 bigint COMMENT 'Consumo total en m3',
consumo_promedio_m3 double COMMENT 'consumo_total_m3 / num_suscriptores',
valor_cargo_fijo double,
valor_consumo double,
valor_total_facturado double,
tarifa_media_m3 double COMMENT 'valor_consumo / consumo_total_m3',
fecha_carga timestamp COMMENT 'Fecha y hora de carga en golden'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_facturacion_periodo"
COMMENT 'KPIs de facturación y consumo por periodo, estrato y concepto'

In [0]:
-- KPI DE RECAUDO POR PERIODO

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.golden.kpi_recaudo_periodo (
periodo integer COMMENT 'Periodo facturado que se paga (yyyyMM)',
tipo_recaudo string COMMENT 'CORRIENTE si el periodo existe en facturación cargada, CARTERA si es de periodos anteriores',
entidad string,
descripcion_entidad string,
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date,
num_pagos bigint,
num_suscriptores bigint COMMENT 'Suscriptores distintos que pagaron',
valor_recaudado double,
pago_promedio double COMMENT 'valor_recaudado / num_pagos',
fecha_carga timestamp COMMENT 'Fecha y hora de carga en golden'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_recaudo_periodo"
COMMENT 'KPIs de recaudo por periodo, entidad, punto de recaudo y fecha de pago'

In [0]:
--KPI CARTERA POR SUSCRIPTOR

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.golden.kpi_cartera_suscriptor (
suscriptor integer,
periodo integer COMMENT 'Periodo facturado (yyyyMM)',
estrato integer,
valor_facturado double,
valor_pagado double,
saldo_pendiente double COMMENT 'valor_facturado - valor_pagado',
num_pagos bigint,
fecha_ultimo_pago date,
estado_pago string COMMENT 'AL_DIA, PAGO_PARCIAL, SIN_PAGO, SALDO_A_FAVOR, RECUPERACION_CARTERA',
fecha_carga timestamp COMMENT 'Fecha y hora de carga en golden'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_cartera_suscriptor"
COMMENT 'Estado de cuenta por suscriptor y periodo: facturado vs pagado'

In [0]:
-- KPI INDICE DE RECAUDO

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.golden.kpi_indice_recaudo (
periodo integer COMMENT 'Periodo facturado (yyyyMM)',
estrato integer,
num_suscriptores_facturados bigint,
num_suscriptores_con_pago bigint,
num_suscriptores_morosos bigint COMMENT 'Suscriptores con SIN_PAGO o PAGO_PARCIAL',
valor_facturado double,
valor_recaudado double,
saldo_pendiente double,
indice_recaudo double COMMENT 'valor_recaudado / valor_facturado',
porcentaje_morosidad double COMMENT 'num_suscriptores_morosos / num_suscriptores_facturados',
fecha_carga timestamp COMMENT 'Fecha y hora de carga en golden'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_indice_recaudo"
COMMENT 'Índice de recaudo y morosidad por periodo y estrato'

In [0]:
-- KPI CALIDAD DE DATOS

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.golden.kpi_calidad_datos (
tabla_origen string COMMENT 'facturacion o recaudos',
periodo integer,
motivo_rechazo string COMMENT 'Regla de calidad incumplida (null = fila resumen sin motivo)',
registros_bronze bigint,
registros_validos bigint,
registros_rechazados bigint,
porcentaje_rechazo double COMMENT 'registros_rechazados / registros_bronze',
fecha_validacion timestamp COMMENT 'Fecha de la validación en silver',
fecha_carga timestamp COMMENT 'Fecha y hora de carga en golden'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpi_calidad_datos"
COMMENT 'Indicadores de calidad de datos por tabla, periodo y motivo de rechazo'

In [0]:
%sql
SHOW TABLES IN catalog_pysmartdata.golden

database,tableName,isTemporary
golden,kpi_calidad_datos,false
golden,kpi_cartera_suscriptor,false
golden,kpi_facturacion_periodo,false
golden,kpi_indice_recaudo,false
golden,kpi_recaudo_periodo,false
